In [1]:
from tensorflow.keras.layers import Input, Embedding, Conv1D, BatchNormalization, Activation, AveragePooling1D, LSTM, Add, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import Initializer, GlorotNormal, Orthogonal
import tensorflow as tf

# ---- custom bias initializer: sets LSTM gates [i, f, c, o] -> [0, forget_bias, 0, 0] ----
class ForgetBiasInitializer(Initializer):
    def __init__(self, forget_bias=2.0):
        self.forget_bias = float(forget_bias)

    def __call__(self, shape, dtype=None):
        # shape is (4 * units,)
        units = shape[0] // 4
        dtype = dtype or tf.float32
        zeros = tf.zeros((units,), dtype=dtype)
        fgate = tf.fill((units,), tf.cast(self.forget_bias, dtype))
        # Keras LSTM gate order: [input, forget, cell, output]
        return tf.concat([zeros, fgate, zeros, zeros], axis=0)

    def get_config(self):
        return {"forget_bias": self.forget_bias}


def build_model(
    input_dim,            # embedding size
    out_dim,              # channels after conv3
    num_layers=1,
    pc_hash_bits=12,
    history_length=1,
    lstm_units=512,
    forget_bias=2.0,
    use_batchnorm=False,
    use_pool=False
):
    vocab_size = (1 << pc_hash_bits) + 1

    inp = Input(shape=(history_length,), dtype="int32", name="seq")
    x = Embedding(vocab_size, input_dim, name="embedding")(inp)

    # conv stack (channels_last: batch, steps, features)
    x = Conv1D(64, 1, name="conv1")(x);  x = Activation("relu", name="relu1")(x)
    x = Conv1D(128, 1, name="conv2")(x); x = Activation("relu", name="relu2")(x)
    x = Conv1D(out_dim, 1, name="conv3")(x); x = Activation("relu", name="relu3")(x)

    if use_batchnorm:
        x = BatchNormalization(name="bn")(x)
    if use_pool:
        x = AveragePooling1D(pool_size=2, name="pool")(x)

    # LSTM(s) with Xavier normal + Orthogonal + custom forget bias
    kernel_init = GlorotNormal()
    recur_init  = Orthogonal()
    bias_init   = ForgetBiasInitializer(forget_bias)

    for i in range(num_layers):
        x = LSTM(
            lstm_units,
            return_sequences=(i < num_layers - 1),
            kernel_initializer=kernel_init,       # Xavier normal
            recurrent_initializer=recur_init,     # Orthogonal
            bias_initializer=bias_init,           # sets forget gate
            unit_forget_bias=False,               # must be False to honor bias_initializer
            name=f"lstm_{i+1}"
        )(x)

    x = Dense(128, name="fc1")(x); x = Activation("relu", name="relu_fc")(x)
    out = Dense(1, activation="tanh", name="out")(x)

    return Model(inp, out, name="BranchNet_TF")

model = build_model(input_dim=64, out_dim=256, num_layers=1, pc_hash_bits=12, history_length=582)
model.compile(optimizer="adam", loss="mse")
model.summary()


2025-08-12 15:43:41.537831: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-12 15:43:41.582071: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-12 15:43:41.582106: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-12 15:43:41.583333: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-12 15:43:41.590065: I tensorflow/core/platform/cpu_feature_guar

Model: "BranchNet_TF"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 seq (InputLayer)            [(None, 582)]             0         
                                                                 
 embedding (Embedding)       (None, 582, 64)           262208    
                                                                 
 conv1 (Conv1D)              (None, 582, 64)           4160      
                                                                 
 relu1 (Activation)          (None, 582, 64)           0         
                                                                 
 conv2 (Conv1D)              (None, 582, 128)          8320      
                                                                 
 relu2 (Activation)          (None, 582, 128)          0         
                                                                 
 conv3 (Conv1D)              (None, 582, 256)         

In [2]:
from dataloader_tf import BranchTraceDatasetTFSingle

# build your model first (history_length must match)
history_length = 28
model = build_model(input_dim=64, out_dim=256, num_layers=1,
                    pc_hash_bits=12, history_length=history_length)

train_ds = BranchTraceDatasetTFSingle(
    trace_paths=['../traces/641.leela_s-602B_dataset.hdf5'],
    br_pc=4320802,
    history_length=history_length,
    pc_bits=32,
    pc_hash_bits=12,
    hash_dir_with_pc=True
).get_dataset(batch_size=128, shuffle=True)

model.compile(optimizer='adam', loss='mse')
model.fit(train_ds, epochs=5, steps_per_epoch=1000)


Epoch 1/5
 122/1000 [==>...........................] - ETA: 5:28 - loss: 0.0672

KeyboardInterrupt: 

In [3]:
# Suppose you have test_traces, test_br_pc, etc.
test_trace_paths = ['../traces/641.leela_s-862B_dataset.hdf5']

test_dataset_loader = BranchTraceDatasetTFSingle(
    trace_paths=test_trace_paths,
    br_pc=4320802,  # use same branch PC or new one as needed
    history_length=history_length,
    pc_bits=32,
    pc_hash_bits=12,
    hash_dir_with_pc=True
)
test_dataset = test_dataset_loader.get_dataset(batch_size=128, shuffle=True)  # batch_size=128

results = model.evaluate(test_dataset)
print('Test loss, Test accuracy:', results)

     21/Unknown - 8s 224ms/step - loss: 3.1589e-06

KeyboardInterrupt: 

In [ ]:
import hls4ml 
import os
import pathlib

os.environ['PATH'] = '/tools/Xilinx/Vivado/2019.1/bin' + os.pathsep + os.environ['PATH']

cfg = hls4ml.utils.config_from_keras_model(model, granularity='name')
cfg['Model'].update({
    'Strategy': 'Resource',
    'IOType': 'io_stream',
    'ReuseFactor': 128,                # modest global RF
    'Precision': 'fixed<16,6>',      # or 'fixed<12,4>' after checking accuracy
})

# Convs: stream-friendly implementation, no extra parallelization
lt = cfg.setdefault('LayerType', {})
lt.setdefault('Conv2D', {})['ConvImplementation'] = 'linebuffer'
lt['Conv2D']['ParallelizationFactor'] = 1

# Override only the heavy layers with valid reuse divisors from the build log
cfg['LayerName'].setdefault('out', {}).update({'Strategy':'Resource', 'ReuseFactor': 128})

# If you can use the Vivado backend, enable FIFO depth optimization:
# cfg['Flows'] = ['vivado:fifo_depth_optimization']
# hls4ml.model.optimizer.get_optimizer('vivado:fifo_depth_optimization') \
#     .configure(profiling_fifo_depth=100_000)

hls_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=cfg, backend='VivadoAccelerator',
    board='pynq-z2', clock_period=10, output_dir='BranchNet_Accel_clean'
)

hls_model.build(csim=False, synth=True, export=True, bitfile=True)


****** Vivado(TM) HLS - High-Level Synthesis from C, C++ and SystemC v2019.1 (64-bit)
  **** SW Build 2552052 on Fri May 24 14:47:09 MDT 2019
  **** IP Build 2548770 on Fri May 24 18:01:18 MDT 2019
    ** Copyright 1986-2019 Xilinx, Inc. All Rights Reserved.

source /tools/Xilinx/Vivado/2019.1/scripts/vivado_hls/hls.tcl -notrace
INFO: [HLS 200-10] Running '/tools/Xilinx/Vivado/2019.1/bin/unwrapped/lnx64.o/vivado_hls'
INFO: [HLS 200-10] For user 'gkapakos' on host 'gkapakos-Type1ProductConfigId' (Linux_x86_64 version 6.8.0-71-generic) on Tue Aug 12 15:45:48 EEST 2025
INFO: [HLS 200-10] On os Ubuntu 24.04.2 LTS
INFO: [HLS 200-10] In directory '/home/gkapakos/Desktop/ECE/10th_Semester/Architecture_of_Parallel_Systems/Project/BranchPredictionAI/BranchNet_Accel_clean'
Sourcing Tcl script 'build_prj.tcl'
INFO: [HLS 200-10] Creating and opening project '/home/gkapakos/Desktop/ECE/10th_Semester/Architecture_of_Parallel_Systems/Project/BranchPredictionAI/BranchNet_Accel_clean/myproject_prj'.
I